In [0]:
%pip install azure-monitor-query
%pip install azure-identity
dbutils.library.restartPython()

In [0]:
config_path = "dbfs:/configs/config.json"
config = spark.read.option("multiline", "true").json(config_path)
first_row = config.first()
env = first_row["env"].strip().lower()
lz_key = first_row["lz_key"].strip().lower()
keyvault_name = f"ingest{lz_key}-meta002-{env}"
client_secret = dbutils.secrets.get(scope=keyvault_name, key='SERVICE-PRINCIPLE-CLIENT-SECRET')
tenant_id = dbutils.secrets.get(scope=keyvault_name, key='SERVICE-PRINCIPLE-TENANT-ID')
client_id = dbutils.secrets.get(scope=keyvault_name, key='SERVICE-PRINCIPLE-CLIENT-ID')
    

In [0]:
from azure.identity import ClientSecretCredential
from azure.monitor.query import LogsQueryClient, LogsQueryStatus
from datetime import datetime, timedelta

credential = ClientSecretCredential(
    tenant_id=tenant_id,
    client_id=client_id,
    client_secret=client_secret
)

client = LogsQueryClient(credential)
workspace_id = "9db45d41-bfe4-49dd-9a3b-5da0ab1a95d0" if lz_key == '00' else '53fbba72-88e5-479c-abb5-ead8d6dbb2f5'

# First, let's see which tables have data in the last 5 hours
query = """
search *
| where TimeGenerated > ago(1d)
| summarize count() by $table
| sort by count_ desc
"""

end_time = datetime.utcnow()
start_time = end_time - timedelta(hours=4)

response = client.query_workspace(
    workspace_id,
    query,
    timespan=(start_time, end_time)
)

if response.status == LogsQueryStatus.SUCCESS:
    print("Tables with data in the last 3 days:\n")
    for table in response.tables:
        for row in table.rows:
            print(f"Table: {row[0]}, Count: {row[1]}")

In [0]:
from azure.identity import ClientSecretCredential
from azure.monitor.query import LogsQueryClient, LogsQueryStatus
from datetime import datetime, timedelta

credential = ClientSecretCredential(
    tenant_id=tenant_id,
    client_id=client_id,
    client_secret=client_secret
)

client = LogsQueryClient(credential)
workspace_id = "9db45d41-bfe4-49dd-9a3b-5da0ab1a95d0" if lz_key == '00' else '53fbba72-88e5-479c-abb5-ead8d6dbb2f5'

# Search AppTraces for your specific message
query = """
AppTraces
| where TimeGenerated > ago(1d)
| where Message contains "CaseNo = "
| project TimeGenerated, Message
| order by TimeGenerated desc
"""

end_time = datetime.utcnow()
start_time = end_time - timedelta(hours=0.5)

response = client.query_workspace(
    workspace_id,
    query,
    timespan=(start_time, end_time)
)

if response.status == LogsQueryStatus.SUCCESS:
    for table in response.tables:
        print(f"Found {len(table.rows)} matching messages\n")
        
        for i, row in enumerate(table.rows[:100]):
            timestamp = row[0]
            message = row[1]
            print(f"\nMessage {i+1}")
            print(f"Time: {timestamp}")
            print(f"Message: {message}")
else:
    print(f"Query failed: {response.status}")

In [0]:
import pandas as pd
from datetime import datetime, timedelta

start_time = datetime(2026, 4, 22, 8, 0, 0)
end_time   = datetime(2026, 4, 22, 10, 0, 0)

query_validate_response = """
AppTraces
| where TimeGenerated between (datetime(2026-04-22 08:00:00) .. datetime(2026-04-22 10:00:00))
| where Message contains "CaseNo = "
| project TimeGenerated, Message, SeverityLevel
| order by TimeGenerated desc
"""

response = client.query_workspace(
    workspace_id,
    query_validate_response,
    timespan=(start_time,end_time)
)

if response.status == LogsQueryStatus.SUCCESS:
    for table in response.tables:
        df_validate_response = pd.DataFrame(table.rows, columns=table.columns)
else:
    print(f"Query failed: {response.status}")
    df_validate_response = pd.DataFrame()

print("Count of Total SBAILS records sent to ARM: ", df_validate_response.count())
print("Count of Successful SBAILS records sent to ARM: ", df_validate_response[df_validate_response['SeverityLevel'] == 1].count())


In [0]:
import pandas as pd
from datetime import datetime, timedelta

start_time = datetime(2026, 4, 21, 15, 0, 0)
end_time   = datetime(2026, 4, 21, 19, 0, 0)

query_validate_response = """
AppTraces
| where TimeGenerated between (datetime(2026-04-21 15:00:00) .. datetime(2026-04-21 20:00:00))
| where Message contains "CaseNo = "
| project TimeGenerated, Message, SeverityLevel
| order by TimeGenerated desc
"""

response = client.query_workspace(
    workspace_id,
    query_validate_response,
    timespan=(start_time,end_time)
)

if response.status == LogsQueryStatus.SUCCESS:
    for table in response.tables:
        df_validate_response = pd.DataFrame(table.rows, columns=table.columns)
else:
    print(f"Query failed: {response.status}")
    df_validate_response = pd.DataFrame()

print("Count of Total ARIAFTA records sent to ARM: ", df_validate_response.count())
print("Count of Successful ARIAFTA records sent to ARM: ", df_validate_response[df_validate_response['SeverityLevel'] == 1].count())


In [0]:
import pandas as pd
from datetime import datetime, timedelta

start_time = datetime(2026, 4, 22, 7, 0, 0)
end_time   = datetime(2026, 4, 22, 9, 0, 0)

query_validate_response = """
AppTraces
| where TimeGenerated between (datetime(2026-04-22 07:00:00) .. datetime(2026-04-22 09:00:00))
| where Message contains "CaseNo = "
| project TimeGenerated, Message, SeverityLevel
| order by TimeGenerated desc
"""

response = client.query_workspace(
    workspace_id,
    query_validate_response,
    timespan=(start_time,end_time)
)

if response.status == LogsQueryStatus.SUCCESS:
    for table in response.tables:
        df_validate_response = pd.DataFrame(table.rows, columns=table.columns)
else:
    print(f"Query failed: {response.status}")
    df_validate_response = pd.DataFrame()

print("Count of Total ARIAUTA records sent to ARM: ", df_validate_response.count())
print("Count of Successful ARIAUTA records sent to ARM: ", df_validate_response[df_validate_response['SeverityLevel'] == 1].count())


In [0]:
import pandas as pd
from datetime import datetime, timedelta

start_time = datetime(2026, 4, 21, 14, 0, 0)
end_time   = datetime(2026, 4, 21, 15, 0, 0)

query_validate_response = """
AppTraces
| where TimeGenerated between (datetime(2026-04-21 14:00:00) .. datetime(2026-04-21 15:00:00))
| where Message contains "CaseNo = "
| project TimeGenerated, Message, SeverityLevel
| order by TimeGenerated desc
"""

response = client.query_workspace(
    workspace_id,
    query_validate_response,
    timespan=(start_time,end_time)
)

if response.status == LogsQueryStatus.SUCCESS:
    for table in response.tables:
        df_validate_response = pd.DataFrame(table.rows, columns=table.columns)
else:
    print(f"Query failed: {response.status}")
    df_validate_response = pd.DataFrame()

print("Count of Total ARIAFPA records sent to ARM: ", df_validate_response.count())
print("Count of Successful ARIAFPA records sent to ARM: ", df_validate_response[df_validate_response['SeverityLevel'] == 1].count())


In [0]:
import pandas as pd
from datetime import datetime, timedelta

start_time = datetime(2026, 4, 21, 8, 0, 0)
end_time   = datetime(2026, 4, 21, 11, 0, 0)

query_validate_response = """
AppTraces
| where TimeGenerated between (datetime(2026-04-21 08:00:00) .. datetime(2026-04-21 11:00:00))
| where Message contains "CaseNo = "
| project TimeGenerated, Message, SeverityLevel
| order by TimeGenerated desc
"""

response = client.query_workspace(
    workspace_id,
    query_validate_response,
    timespan=(start_time,end_time)
)

if response.status == LogsQueryStatus.SUCCESS:
    for table in response.tables:
        df_validate_response = pd.DataFrame(table.rows, columns=table.columns)
else:
    print(f"Query failed: {response.status}")
    df_validate_response = pd.DataFrame()

print("Count of Total Bails records sent to ARM: ", df_validate_response.count())
print("Count of Successful Bails records sent to ARM: ", df_validate_response[df_validate_response['SeverityLevel'] == 1].count())


In [0]:
# dbutils.fs.rm("dbfs:/checkpoints/APPEALS/ARCHIVE/", recurse=True)
# dbutils.fs.rm("dbfs:/checkpoints/ARCHIVE/", recurse=True)